# 📝 과제 LV2(응용): 기존 챗봇·RAG 시스템 연동

지난 단원에서 만든 시스템을 UI에 **연결**합니다. RAG·LLM 로직은 **새로 만들지 않습니다.** `core` 의 함수를 import 해서 화면에만 얹으세요.

| | |
| --- | --- |
| 데이터 | `data/taxis.csv` (뉴욕 택시 운행 6,433건) |
| 실행 | 루트에서 `uv run streamlit run 과제_LV2_응용/app.py` |
| 화면 | 탭 3개 (챗봇 · 문서 Q&A · 택시 대시보드) |
| 키 | `.streamlit/secrets.toml` 의 `OPENAI_API_KEY` (없으면 앱이 안내를 띄우고 멈춥니다) |

`app.py` 의 같은 번호 주석 자리에 코드를 채우세요. **탭 3개와 사이드바 틀은 이미 제공**되어 있으니 그 안을 채우면 됩니다.


### 제공되는 `core` 함수 (재작성 금지)

```text
chatbot_core.stream_reply(message, history) -> 답변 토큰 스트리밍 (st.write_stream 에 넘긴다)
rag_core.ask(question, k)                   -> {"answer": 답변, "sources": [{title, text, score}, ...]}
keys.require_openai_key_or_stop()           -> 키가 없으면 안내를 띄우고 앱을 멈춘다 (app.py 에 이미 제공)
```

- `history` 형식: `[{"role": "user"|"assistant", "content": ...}]`
- 키 확인은 `require_openai_key_or_stop()` 한 줄이 이미 처리합니다. 여러분 코드에서 키를 다시 검사하지 마세요.


### 완성 화면

**💬 챗봇 탭** (문제 1~3)

![LV2 챗봇 탭](../images/lv2/lv2_chat.png)

**📚 문서 Q&A 탭** (문제 4~5)

![LV2 문서 Q&A 탭](../images/lv2/lv2_rag.png)

**📊 택시 대시보드 탭** (문제 7~8)

![LV2 택시 대시보드 탭](../images/lv2/lv2_dashboard.png)


## 1. 대화 이력 준비와 표시

- `st.session_state.messages` 를 빈 리스트로 초기화. **키가 없을 때만**
- 저장된 이력을 `for` 로 순회하며 `st.chat_message(role)` 안에 내용 표시

**확인**: 다른 탭에 갔다 와도 이전 대화가 말풍선으로 다시 보인다.


## 2. 새 입력 받기 (`st.chat_input`)

- `st.chat_input` 으로 입력을 받는다 (입력이 없으면 `None` 이므로 있을 때만 처리)
- 받은 입력을 `{"role": "user", "content": ...}` 로 이력에 추가하고 `st.chat_message("user")` 로 표시

**확인**: 입력창에 치면 사용자 말풍선에 그 문장이 뜬다.


## 3. 스트리밍 답변 연동 (`st.write_stream`)

- `st.chat_message("assistant")` 안에서 `st.write_stream(chatbot_core.stream_reply(prompt, 이력))` 호출
- 넘기는 이력에서 **방금 추가한 내 메시지는 제외**한다 (그 문장은 첫 번째 인자로 이미 갑니다. 빼지 않으면 모델이 같은 말을 두 번 받습니다)
- `st.write_stream` 이 돌려준 완성 답변을 `{"role": "assistant", ...}` 로 이력에 추가

**확인**: 답변이 한 조각씩 타이핑되듯 나타나고, 재실행 후에도 이력에 남는다.


## 4. RAG 답변 표시 (`rag_core.ask`)

- `st.text_input` 으로 질문을 받는다
- **질문이 비어 있지 않을 때만** `rag_core.ask(question, k=top_k)` 호출 (빈 문자열에도 부르면 화면을 열자마자 요금이 나갑니다)
- 결과의 `answer` 를 `st.write` 로 표시

**확인**: "파일 용량 제한이 얼마인가요?" 를 물으면 관련 안내 문장이 답으로 나온다.


## 5. 근거 문서 표시 (`st.expander`)

- `result["sources"]` 를 순회하며 문서마다 `st.expander` 를 연다
- expander 라벨은 `제목 (유사도 점수)` 형태, 안에는 본문(`text`)

**확인**: 답변 아래에 근거 문서 목록이 접힌 채로 나오고, 펼치면 원문이 보인다.


## 6. 사이드바 컨트롤

- `st.slider` 로 `top_k`(RAG 참고 문서 수) 를 받는다. 범위 1~5, 기본 3 (스켈레톤에 박아 둔 상수를 이 값으로 교체)
- `st.button("대화 초기화")` 를 누르면 `st.session_state.messages` 를 비우고 `st.rerun()`
- 다른 탭이 참조하므로 **사이드바를 위쪽에서 먼저** 만든다

**확인**: 슬라이더를 4로 올리면 문서 Q&A 가 문서 4개를 참고하고, 초기화 버튼으로 대화가 비워진다.


## 7. 데이터 캐싱 로딩 (`@st.cache_data`)

- `@st.cache_data` 를 붙인 함수로 `taxis.csv` 를 읽어 DataFrame 반환
- 경로는 문자열로 적지 말고 제공된 `DATA_DIR` 에 파일명을 이어 붙인다

**확인**: 탭을 오갈 때 데이터가 즉시 나온다(재로딩 없음).


## 8. 지표와 차트 (`st.metric` + plotly)

- `st.columns(3)` + `st.metric` 으로 3개: 운행 건수(`len`, 천단위 쉼표) · 평균 요금(`fare` 평균, `$` 소수 2자리) · 평균 팁(`tip` 평균, `$` 소수 2자리)
- `px.bar` → `st.plotly_chart` 로 **결제 수단별 운행 건수** (`payment` 의 `value_counts()`)
- `px.bar` → `st.plotly_chart` 로 **승차 자치구별 평균 요금** (`pickup_borough` 로 group 후 `fare` 평균)
- `value_counts()`·`groupby()` 결과는 Series 이므로 `reset_index(name="건수")` 처럼 **열 이름을 붙여야** `px.bar` 의 x·y 에 쓸 수 있다

**확인**: `운행 건수 6,433건 · 평균 요금 $13.09 · 평균 팁 $1.98` 지표와 막대그래프 2개가 보인다.
